# Configuration

In [1]:
import os 
import pickle 

if True ^ os.getcwd().endswith('hte-and-targeting'):
    os.chdir('..')

from typing import Union, List, Iterable

In [2]:
import random
import pandas as pd 
import numpy as np

from tqdm import tqdm

In [3]:
from statsmodels.regression.linear_model import OLS

In [4]:
# visualization
import matplotlib.pyplot as plt
import seaborn as sns

# visualization params
# label size
tick_label_size = 12
legend_label_size = 12
axis_label_size = 14
title_size = 18
# font
plt.rcParams['font.family'] = 'serif'

In [5]:
from core.variables import *
from core.dgp import SingleSegment

# DGP

In [6]:
dgp_params = {
    'te': 0.6,
    'noise_std': 2,
    'treatment_space': np.array([0, 1, 2, 3, 4])
}

dgp = SingleSegment(**dgp_params)
treatment_arr, outcome_arr = dgp.sample(sample_size=1000, seed=0)

# Demand Model

In [7]:
def difference_in_mean(
        treatments: np.ndarray, outcomes: np.ndarray, 
        treatment_vals: Iterable, 
) -> np.ndarray:
    """ 
    Calculate lift for each treatment value compared to the control group 
    using the difference in mean.

    Params:
    -------
    treatments: np.ndarray
        The treatment values.
    outcomes: np.ndarray
        The outcomes.
    treatment_vals: Iterable
        All treatment levels

    Returns:
    --------
    np.ndarray
        The lift for each treatment value compared to the control group. 
    """
    assert treatment_vals[0] == 0, "The first treatment value should be the control group."

    # placeholder for the treatment effect estimates
    te_arr = np.zeros(treatment_vals.shape[0])

    # mask for control group
    control_mask = treatments == 0

    # iterate over the treatment values
    for idx, treatment_val in enumerate(treatment_vals):
        # mask for the treatment value
        treatment_mask = treatments == treatment_val
        
        # calculate the difference in mean
        te_arr[idx] = outcomes[treatment_mask].mean() - outcomes[control_mask].mean()

    return te_arr

In [9]:
emp_te_arr = difference_in_mean(
    treatments=treatment_arr, outcomes=outcome_arr,
    treatment_vals=dgp_params['treatment_space']
)

# display
for idx, te_est in enumerate(emp_te_arr):
    if idx == 0:
        continue 
    
    print(f'Treatment value: {dgp.treatment_space[idx]}, Estiamted TE: {te_est:.4f}, True TE: {dgp.lift_arr[idx]:.4f}')

Treatment value: 1, Estiamted TE: 0.6710, True TE: 0.6000
Treatment value: 2, Estiamted TE: 1.0295, True TE: 1.2000
Treatment value: 3, Estiamted TE: 1.9261, True TE: 1.8000
Treatment value: 4, Estiamted TE: 2.3537, True TE: 2.4000


# Optimization

In [18]:
def optimize(
    customer_tes: np.ndarray, treatment_space: np.ndarray,
    price: float, cost: float, 
) -> tuple:
    """
    Optimize the treatment assignment for a given price and cost.

    Params:
    -------
    customer_tes: np.ndarray, shape = (n_treatments - 1, )
        Lift under each treatment level except control
    treatment_space: np.ndarray, shape = (n_treatments, )
        The treatment space.
    price: float
        The price.
    cost: float
        The cost.

    Returns:
    --------
    (opt_decision, opt_obj): tuple
        - opt_decision: int
            The optimal treatment decision.
        - opt_obj: float
            The optimal objective value.
    """
    assert treatment_space[0] == 0, "The first treatment value should be the control group."    
    
    # calculate the profit lift for each customer and each treatment
    profit_lift_arr = customer_tes * price - treatment_space * cost 

    # optimize
    opt_decision = treatment_space[profit_lift_arr.argmax()]
    opt_obj = profit_lift_arr.max().sum()

    return opt_decision, opt_obj

def obj_func(
    customer_tes: np.ndarray, targ_decision: int, 
    price: float, cost: float, 
) -> float:
    """
    Objective function for the optimization problem.

    Params:
    -------
    customer_tes: np.ndarray, shape = (n_treatments - 1, )
        Lift under each treatment level except control
    targ_decision: int
        The treatment decision.
    price: float
        The price.
    cost: float
        The cost.

    Returns:
    --------
    float
        The objective value.
    """
    return customer_tes[targ_decision] * price - targ_decision * cost

In [21]:
plugin_decision, plugin_val_est = optimize(
    customer_tes=emp_te_arr, 
    treatment_space=dgp_params['treatment_space'],
    price=1, cost=0.5
)

true_plugin_val = obj_func(
    customer_tes=dgp.lift_arr, 
    targ_decision=plugin_decision,
    price=1, cost=0.5
)

print(f'Estimated targing value: {plugin_val_est:.4f}, True targeting value: {true_plugin_val:.4f}')

Estimated targing value: 0.4261, True targeting value: 0.3000


# Winner's Curse

In [22]:
def repeated_experiments(
    operations_params: dict, dgp_params: dict, data_params: dict, 
    experiment_params: dict, estimators_dict: dict
) -> list:
    pass 